In [29]:
#!/usr/bin/env python3
"""
KAORU BRIDGE v48.0 - THE C++ LEGACY
"Hackeando el RNG de Microsoft Visual C++ (MSVC) de 2009"
"""

import hashlib
import time
import sys
from datetime import datetime

class KaoruLegacyRNG:

    SATOSHI_ADDR = "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"
    GENESIS_TIME = 1231006505

    # --- MATEMÁTICA SECP256K1 ---
    P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
    G_X = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
    G_Y = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

    def modinv(self, a, m): return pow(a, m - 2, m)

    def point_add(self, P1, P2):
        if P1 is None: return P2
        if P2 is None: return P1
        x1, y1 = P1
        x2, y2 = P2
        if x1 == x2 and y1 != y2: return None
        if x1 == x2: m = (3 * x1 * x1) * self.modinv(2 * y1, self.P)
        else: m = (y1 - y2) * self.modinv(x1 - x2, self.P)
        x3 = (m * m - x1 - x2) % self.P
        y3 = (m * (x1 - x3) - y1) % self.P
        return (x3, y3)

    def scalar_mul(self, k, Point):
        R = None
        for i in range(256):
            if (k >> i) & 1: R = self.point_add(R, Point)
            Point = self.point_add(Point, Point)
        return R

    def get_address(self, pub_bytes):
        sha = hashlib.sha256(pub_bytes).digest()
        try:
            import Crypto.Hash.RIPEMD160 as R
            h = R.new()
            h.update(sha)
            ripemd = h.digest()
        except:
            return "ERROR_LIB" # Necesitas pycryptodome

        version = b'\x00' + ripemd
        checksum = hashlib.sha256(hashlib.sha256(version).digest()).digest()[:4]
        payload = version + checksum

        alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
        val = int.from_bytes(payload, 'big')
        res = ""
        while val > 0:
            val, mod = divmod(val, 58)
            res = alphabet[mod] + res
        for b in payload:
            if b == 0: res = "1" + res
            else: break
        return res

    # --- EL RNG DE MICROSOFT (MSVC) ---
    class MSVCRand:
        def __init__(self, seed):
            self.state = seed

        def rand(self):
            # Fórmula exacta de MSVC: state * 214013 + 2531011 (32-bit overflow)
            self.state = (self.state * 214013 + 2531011) & 0xFFFFFFFF
            # Retorna bits 16-30
            return (self.state >> 16) & 0x7FFF

    def generate_key_from_msvc(self, seed):
        """Genera 32 bytes usando rand() como se haría en C++"""
        rng = self.MSVCRand(seed)
        key_bytes = bytearray()

        for _ in range(32):
            # Obtener un byte aleatorio (rand() % 256)
            r = rng.rand()
            byte = r & 0xFF
            key_bytes.append(byte)

        return int.from_bytes(key_bytes, 'big')

    def execute(self, window_hours=2):
        print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║               KAORU BRIDGE v48.0 - THE C++ LEGACY                    ║
║           "Hackeando el RNG de Microsoft Visual C++ 2009"            ║
╚══════════════════════════════════════════════════════════════════════╝
        """)

        try:
            import Crypto.Hash.RIPEMD160
        except:
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "pycryptodome"])

        start_time = self.GENESIS_TIME - (window_hours * 3600)
        end_time = self.GENESIS_TIME + (window_hours * 3600)

        print(f"   [1] ⏳ Simulando entropía de Windows XP...")
        print(f"       Desde: {datetime.fromtimestamp(start_time)}")
        print(f"       Hasta: {datetime.fromtimestamp(end_time)}")

        for t in range(start_time, end_time):
            # 1. Generar clave usando el algoritmo de MSVC
            k = self.generate_key_from_msvc(t)

            # Chequeo rápido solo para el log (1 de cada 1000)
            if t % 1000 == 0:
                print(f"   scanning... {datetime.fromtimestamp(t)} | Seed: {t}", end="\r")

            # Si es el momento exacto, mostramos detalle
            if t == self.GENESIS_TIME:
                print(f"\n\n   🎯 ¡SEED DEL GÉNESIS! ({t})")
                self.check_candidate(k, t)

            # Verificar TODOS (silenciosamente si no es match)
            # Para optimizar, solo hacemos la multiplicación escalar completa
            # si los primeros bytes coinciden con una optimización (pero aquí haremos fuerza bruta pura)
            # (En Python esto es lento, así que solo chequeamos el Genesis exacto en detalle)
            # Si quieres chequear TODOS, necesitarías C++.

            # Para la demo, asumimos que Satoshi sincronizó su reloj
            # y chequeamos solo el momento exacto y alrededores cercanos en detalle
            if abs(t - self.GENESIS_TIME) < 5:
                 self.check_candidate(k, t)

        print(f"\n\n   [2] 🏁 Simulación de C++ finalizada.")

    def check_candidate(self, k, seed):
        k_hex = hex(k)[2:].zfill(64)

        # Derivar
        pub_point = self.scalar_mul(k, (self.G_X, self.G_Y))
        if not pub_point: return

        # Satoshi 04 (Uncompressed)
        pub_bytes = b'\x04' + pub_point[0].to_bytes(32, 'big') + pub_point[1].to_bytes(32, 'big')
        address = self.get_address(pub_bytes)

        if seed == self.GENESIS_TIME:
            print(f"   🔑 Clave MSVC: {k_hex}")
            print(f"   📬 Dirección:  {address}")

        if address == self.SATOSHI_ADDR:
            print("\n   🚨🚨🚨 ¡¡¡LO TENEMOS!!! 🚨🚨🚨")
            print(f"   SATOSHI USÓ 'srand({seed})' EN WINDOWS!!!")
            sys.exit()

if __name__ == "__main__":
    bridge = KaoruLegacyRNG()
    bridge.execute()


╔══════════════════════════════════════════════════════════════════════╗
║               KAORU BRIDGE v48.0 - THE C++ LEGACY                    ║
║           "Hackeando el RNG de Microsoft Visual C++ 2009"            ║
╚══════════════════════════════════════════════════════════════════════╝
        
   [1] ⏳ Simulando entropía de Windows XP...
       Desde: 2009-01-03 16:15:05
       Hasta: 2009-01-03 20:15:05
   scanning... 2009-01-03 18:06:40 | Seed: 1231006000

   🎯 ¡SEED DEL GÉNESIS! (1231006505)
   🔑 Clave MSVC: f969521e954fb209f50f133b95c181a678f2a4602599e43ef04cd0a47ab4c19b
   📬 Dirección:  1AUdytu4ud2KeHaBS7WnTD7JmDq3tvxPMX
   🔑 Clave MSVC: f969521e954fb209f50f133b95c181a678f2a4602599e43ef04cd0a47ab4c19b
   📬 Dirección:  1AUdytu4ud2KeHaBS7WnTD7JmDq3tvxPMX
   scanning... 2009-01-03 20:03:20 | Seed: 1231013000

   [2] 🏁 Simulación de C++ finalizada.
